In [ ]:
# Crescent Bakery: Correlation and A/B Testing

Lesson 1.7 Guided Example. Computes correlations on the bakery dataset,
demonstrates a confounded correlation in synthetic data, simulates an A/B
test, and walks through power calculations.

Author: Paola Paguaga
Date: Sept 2, 2026

In [8]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px

In [9]:
bakery = pd.read_csv("../lesson-1-2-types-of-data/bakery_customers.csv")

#Pearson correlation between total spent and visits

pearson_r, pearson_p = stats.pearsonr(
    bakery["total_spent_usd"],
    bakery["visits_last_year"],
)

#Spearman correlation between the same two columns

spearman_r, spearman_p = stats.spearmanr(
    bakery["total_spent_usd"],
    bakery["visits_last_year"],
)

print(f"Pearson r:  {pearson_r:.3f} (p = {pearson_p:.4f})")
print(f"Spearman r: {spearman_r:.3f} (p = {spearman_p:.4f})")

Pearson r:  0.182 (p = 0.2071)
Spearman r: 0.150 (p = 0.2982)


In [20]:
fig = px.scatter(
    bakery,
    x="total_spent_usd",
    y="visits_last_year",
    title="Total Spent vs Visits in the Last Year",
    trendline="ols",
)
fig.show("browser")

In [27]:
np.random.seed(42)

n = 500

# Z is the underlying "engagement level" (the confounder)
engagement = np.random.normal(loc=0, scale=1, size=n)

# X depends on engagement (more engaged users are more likely to use chat)
chat_usage_propensity = engagement + np.random.normal(loc=0, scale=0.5, size=n)

# Y depends on engagement (more engaged users are more likely to convert)
conversion_propensity = engagement + np.random.normal(loc=0, scale=0.5, size=n)

# Make X and Y binary outcomes
chat_used = (chat_usage_propensity > 0).astype(int)
converted = (conversion_propensity > 0).astype(int)

confounded = pd.DataFrame({
    "chat_used": chat_used,
    "converted": converted,
    "engagement": engagement,
})

# Compute the correlation between chat_used and converted
r, p = stats.pearsonr(confounded["chat_used"], confounded["converted"])
print(f"Correlation between chat_used and converted: r = {r:.3f}, p = {p:.4f}")

# And the conversion rates by chat usage
conversion_rates = confounded.groupby("chat_used")["converted"].agg(["mean", "count"])
print("\nConversion rate by chat usage:")
print(conversion_rates)

Correlation between chat_used and converted: r = 0.624, p = 0.0000

Conversion rate by chat usage:
               mean  count
chat_used                 
0          0.201613    248
1          0.825397    252


In [28]:
np.random.seed(43)

n = 500

# Same engagement distribution as before
engagement = np.random.choice([0,1], size=n)

# Random assignment to treatment (chat enabled) or control
treatment = np.random.choice([0,1], size=n)

# Conversion still depends on engagement, but now ALSO has a small NEGATIVE
# treatment effect (chat slightly hurts conversion, like Hannah's real result)
true_treatment_effect = -0.2 # in z-scores units

conversion_propensity = engagement + true_treatment_effect * treatment + np.random.normal(loc=0, scale=0.5, size=n)
converted = (conversion_propensity > 0).astype(int)

ab_test = pd.DataFrame({
    "treatment": treatment,
    "converted": converted,
})

# Conversion rates by treatment group
ab_results = ab_test.groupby("treatment")["converted"].agg(["mean", "count"])
ab_results.index = ["Control", "Treatment"]
print("A/B test conversion rates:")
print(ab_results)

# Two-proportion z-test for the difference
control_conv = ab_test[ab_test["treatment"] == 0]["converted"]
treatment_conv = ab_test[ab_test["treatment"] == 1]["converted"]

n_control = len(control_conv)
n_treatment = len(treatment_conv)
p_control = control_conv.mean()
p_treatment = treatment_conv.mean()

# Pooled proportion for the test
p_pool = (control_conv.sum() + treatment_conv.sum()) / (n_control + n_treatment)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treatment))
z = (p_treatment - p_control) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

# 95% CI for the difference in proportions
se_diff = np.sqrt(
    p_control * (1 - p_control) / n_control
    + p_treatment * (1 - p_treatment) / n_treatment
)
diff = p_treatment - p_control
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print(f"\nTreatment effect: {diff:.3f}")
print(f"95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")
print(f"z = {z:.3f}, p = {p_value:.4f}")

A/B test conversion rates:
               mean  count
Control    0.701613    248
Treatment  0.646825    252

Treatment effect: -0.055
95% CI: (-0.137, 0.027)
z = -1.307, p = 0.1913


In [29]:
# Inputs
p1 = 0.08
p2 = 0.09
alpha = 0.05
power = 0.80

# Critical values
z_alpha_2 = stats.norm.ppf(1 - alpha/2)  # 1.96 for two-sided alpha=0.05
z_beta = stats.norm.ppf(power)            # 0.84 for power=0.80

# Sample size formula for comparing two proportions
delta = abs(p2 - p1)
variance_term = p1 * (1 - p1) + p2 * (1 - p2)
n_per_group = ((z_alpha_2 + z_beta) ** 2 * variance_term) / delta ** 2

print(f"To detect a lift from {p1:.1%} to {p2:.1%} at {1-alpha:.0%} confidence with {power:.0%} power:")
print(f"  Required sample size per group: {n_per_group:.0f}")
print(f"  Total sample size: {2 * n_per_group:.0f}")

To detect a lift from 8.0% to 9.0% at 95% confidence with 80% power:
  Required sample size per group: 12205
  Total sample size: 24410


In [30]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Compute the standardized effect size for two proportions (Cohen's h)
effect_size = proportion_effectsize(p2, p1)

# Solve for sample size
power_analysis = NormalIndPower()
n_statsmodels = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    alternative="two-sided",
)

print(f"Cohen's h effect size: {effect_size:.4f}")
print(f"Required n per group (statsmodels): {n_statsmodels:.0f}")

Cohen's h effect size: 0.0359
Required n per group (statsmodels): 12199


In [31]:
effects_to_test = [0.005, 0.01, 0.02, 0.04]  # absolute lifts from 8%
results = []

for effect in effects_to_test:
    p2_test = 0.08 + effect
    delta = abs(p2_test - 0.08)
    variance_term = 0.08 * 0.92 + p2_test * (1 - p2_test)
    n = ((z_alpha_2 + z_beta) ** 2 * variance_term) / delta ** 2
    results.append({
        "lift_pp": effect * 100,
        "p2_pct": p2_test * 100,
        "n_per_group": int(np.ceil(n)),
    })

results_df = pd.DataFrame(results)
print("How sample size scales with desired effect size:")
print(results_df.to_string(index=False))

How sample size scales with desired effect size:
 lift_pp  p2_pct  n_per_group
     0.5     8.5        47525
     1.0     9.0        12206
     2.0    10.0         3211
     4.0    12.0          880
